# Exercise 2: PyTorch core

In this exercise you’ll build core PyTorch “muscle memory” that you’ll reuse in basically every model you write:

- **Autograd**: how gradients are created, how they accumulate, and how to compute gradients for one or multiple inputs.
- **Dataloading**: writing small `Dataset`s, using `DataLoader`, and custom `collate_fn`.
- **Optimizers**: implementing **AdamW** updates from scratch (state, bias correction, weight decay).
- **Training basics**: a clean single training step.
- **Initialization**: fan-in/out and common initializers (Xavier / Kaiming), plus a helper to init `nn.Linear`.

As before: fill in all `TODO`s without changing function names or signatures.
When debugging, print shapes/dtypes/devices, and write tiny sanity checks (e.g. compare to PyTorch’s built-ins).


In [3]:
from __future__ import annotations
from dataclasses import dataclass
import torch
from torch import nn

## Autograd fundamentals

PyTorch builds a computation graph when you apply operations to tensors with `requires_grad=True`.
Calling `backward()` (or `torch.autograd.grad`) computes gradients by traversing that graph.

### Key concepts
- **Leaf tensor**: a tensor created by you (not the result of an operation) with `requires_grad=True`. Leaf tensors can store gradients in `.grad`.
- **Gradient accumulation**: calling `backward()` adds into `.grad` (it does not overwrite). You must reset gradients between steps/calls.
- **`torch.autograd.grad` vs `.backward()`**
  - `torch.autograd.grad(f, x)` returns `df/dx` directly and does not write into `x.grad` unless you explicitly do so.
  - `f.backward()` writes gradients into `.grad` of leaf tensors.

In the next functions you’ll compute gradients for a simple scalar function such as `f(x) = sum(x^2)` using both APIs.

### `torch.no_grad()`
Wrap inference-only code to avoid tracking gradients and building graphs:
- saves memory
- speeds up evaluation

### `detach()`
`y = x.detach()` returns a tensor that shares data with `x` but is **not connected** to the autograd graph.
This is useful when you want to treat something as a constant target.

### `model.train()` vs `model.eval()`
- `train()` enables training behavior (e.g. dropout active, batchnorm updates running stats).
- `eval()` enables inference behavior (e.g. dropout off, batchnorm uses running stats).

In [4]:
def grad_with_autograd_grad(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using torch.autograd.grad

    Requirements:
    - Do not call .backward().
    - x should require grad inside the function (don't assume it does).
    - Must return df/dx
    """
    x = x.requires_grad_(True)
    f = (x**2).sum()
    return torch.autograd.grad(f,x)
t = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(grad_with_autograd_grad(t))

(tensor([[ 2.,  4.,  6.],
        [ 8., 10., 12.]]),)


/home/sebi/playground_ws/ethz-course-2026/hw1_pytorch_tutorial/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


In [5]:

def grad_with_backward(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using .backward().

    Requirements:
    - Must return df/dx
    - Must not leak gradients across calls (watch x.grad accumulation)
    """
    x = x.clone().detach().requires_grad_(True)
    f = (x**2).sum()
    f.backward()
    return x.grad
t = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(grad_with_backward(t))

tensor([[ 2.,  4.,  6.],
        [ 8., 10., 12.]])


In [6]:
def grad_wrt_multiple_inputs(
    a: torch.Tensor, b: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute gradients w.r.t. multiple inputs. The function is f(a, b) = sum(a^2 + ab).

    Return:
        (df/da, df/db)

    Requirements:
    - Use torch.autograd.grad
    - Ensure both a and b require grad in this function.
    """
    a = a.clone().detach().requires_grad_(True)
    b = b.clone().detach().requires_grad_(True)
    f = (a**2 + a*b).sum()
    grad_a, grad_b = torch.autograd.grad(f, (a, b))
    return grad_a, grad_b
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
b = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(grad_wrt_multiple_inputs(a,b))

(tensor([[ 3.,  6.,  9.],
        [12., 15., 18.]]), tensor([[1., 2., 3.],
        [4., 5., 6.]]))


## Dataloading

In PyTorch, a `Dataset` defines how to fetch a *single* training example, and a `DataLoader` handles:
- batching
- shuffling
- parallel workers
- optional custom batching logic via `collate_fn`

### `Dataset` in one sentence
A `Dataset` only needs:
- `__len__`: number of items
- `__getitem__`: return one item (e.g. `(x, y)`)

### Why `collate_fn` matters
The default DataLoader collation stacks items along a new batch dimension.
That works for fixed-size tensors, but it breaks for **variable-length sequences**.

So we’ll implement padding ourselves:
- Convert a list of 1D token sequences into a padded tensor `(B, T_max)`
- Track `lengths` and a `padding_mask`

### Mask convention for padding
For padding masks in this exercise:
- `padding_mask[b, t] == True` means **this is padding / invalid**
- `padding_mask[b, t] == False` means **this is a real token**

In [7]:
from torch.utils.data import DataLoader, Dataset

In [8]:
class TensorPairDataset(Dataset):
    """
    Minimal dataset wrapping (x, y).

    x: (N, ...)
    y: (N, ...)

    N is the number of samples. The dataset should return tuples of (x[i], y[i]).
    """

    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        if x.shape[0] != y.shape[0]:
            raise ValueError("x and y must have the same amount of samples")
        self.x = x
        self.y = y

    def __len__(self) -> int:
        return self.x.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.x[idx], self.y[idx]

In [9]:
class NextTokenDataset(Dataset):
    """
    Next-token prediction dataset.

    Given tokens of shape (N, T), produce:
      input_ids  = tokens[:, :-1]
      target_ids = tokens[:, 1:]

    Return per item:
      (input_ids, target_ids)

    Notes:
    - Returned tensors should be 1D of length (T-1).
    - dtype should remain integer.
    """

    def __init__(self, tokens: torch.Tensor):
        self.tokens = tokens

    def __len__(self) -> int:
        return self.tokens.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        seq = self.tokens[idx]
        input_ids = seq[:-1]
        target_ids = seq[1:]
        return input_ids, target_ids

In [10]:

class RandomCropSequenceDataset(Dataset):
    """
    Sequence dataset that returns random crops of fixed length.

    tokens: (N, T_total)
    crop_len: L

    For each __getitem__:
      - sample a start index s so that s+L <= T_total
      - return tokens[idx, s:s+L]

    Requirements:
    - Use a torch.Generator for deterministic behavior if seed is provided.
    - Do NOT use Python's random module.
    """

    def __init__(self, tokens: torch.Tensor, crop_len: int, seed: int | None = None):
        self.tokens = tokens
        self.crop_len = crop_len

        self.generator = None
        if seed is not None:
            self.generator = torch.Generator()
            self.generator.manual_seed(seed)
    def __len__(self) -> int:
        return self.tokens.shape[0]

    def __getitem__(self, idx: int) -> torch.Tensor:
        T_total = self.tokens.shape[1]
        num = torch.randint(0, T_total - self.crop_len + 1, (), generator=self.generator).item()
        return self.tokens[idx, num:num+self.crop_len]

In [ ]:


@dataclass(frozen=True)
class PaddedBatch:
    """
    A padded batch for variable-length sequences.

    tokens: LongTensor (B, T_max)
    lengths: LongTensor (B,)
    padding_mask: BoolTensor (B, T_max) where True means "this is padding"
    """

    tokens: torch.Tensor
    lengths: torch.Tensor
    padding_mask: torch.Tensor


def pad_1d_sequences(seqs: list[torch.Tensor], pad_value: int = 0) -> PaddedBatch:
    """
    Pad a list of 1D integer tensors to the same length.

    Requirements:
    - Return PaddedBatch(tokens, lengths, padding_mask)
    - padding_mask[b, t] == True iff t >= lengths[b]
    - tokens should be dtype long, if not cast them
    """
   
    lengths = torch.tensor([s.numel() for s in seqs])
    T_max = lengths.max().item()
    B = len(seqs)
    
    tokens = torch.full((B, T_max), pad_value, dtype= torch.long)

    for i, s in enumerate(seqs):
        tokens[i, :s.numel()] = s.long()

    padding_mask = torch.arange(T_max).unsqueeze(0) >= lengths.unsqueeze(1)
    
    return PaddedBatch(tokens, lengths, padding_mask)
    
seqs = [
    torch.tensor([7, 2, 9]),
    torch.tensor([4, 1]),
    torch.tensor([8, 5, 6, 3]),
]
print(pad_1d_sequences(seqs, pad_value=0))

PaddedBatch(tokens=tensor([[7, 2, 9, 0],
        [4, 1, 0, 0],
        [8, 5, 6, 3]]), lengths=tensor([3, 2, 4]), padding_mask=tensor([[False, False, False,  True],
        [False, False,  True,  True],
        [False, False, False, False]]))


In [36]:
def collate_next_token_batch(
    batch: list[tuple[torch.Tensor, torch.Tensor]], pad_value: int = 0
) -> dict[str, torch.Tensor]:
    """
    Collate for NextTokenDataset samples that may have variable lengths.

    batch: list of (input_ids, target_ids), each 1D

    Return dict with:
      - input_ids: (B, T_max)
      - target_ids: (B, T_max)
      - attention_mask: (B, T_max) where True means "keep" (NOT padding)
      - padding_mask: (B, T_max) where True means "padding"

    Requirements:
    - pad input_ids and target_ids consistently
    - attention_mask is the logical NOT of padding_mask
    """
    input_ids = [x for x,_ in batch]
    target_ids = [y for _,y in batch]

    input_batch = pad_1d_sequences(input_ids, pad_value)
    target_batch = pad_1d_sequences(target_ids, pad_value)

    padding_mask = input_batch.padding_mask
    attention_mask = ~padding_mask
    
    return {
        "input_ids": input_batch.tokens,
        "target_ids": target_batch.tokens,
        "attention_mask": attention_mask,
        "padding_mask": padding_mask
    }
#list [(start, target)]
batch = [
        (
            torch.tensor([5, 0, 2]),
            torch.tensor([0, 2, 9]),
        ),
        (
            torch.tensor([4, 1]),
            torch.tensor([1, 8]),
        ),
    ]
print(collate_next_token_batch(batch))


{'input_ids': tensor([[5, 0, 2],
        [4, 1, 0]]), 'target_ids': tensor([[0, 2, 9],
        [1, 8, 0]]), 'attention_mask': tensor([[ True,  True,  True],
        [ True,  True, False]]), 'padding_mask': tensor([[False, False, False],
        [False, False,  True]])}


In [ ]:
def make_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    drop_last: bool = False,
    collate_fn=None,
    num_workers: int = 0,
) -> DataLoader:
    """
    Create a DataLoader with optional collate_fn.
    """
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=collate_fn,
        drop_last=drop_last)
    

## Optimizers (AdamW from scratch)

PyTorch optimizers keep **state** for each parameter (e.g. moment estimates in Adam).
In this section you’ll implement **AdamW**, which is Adam + *decoupled* weight decay.

### AdamW state
For each parameter tensor `p` we store:
- `m`: first moment (EMA of gradients)
- `v`: second moment (EMA of squared gradients)
- `t`: step counter

### Update overview (high level)
1) Update moments `m, v`
2) Bias-correct them (`m_hat, v_hat`)
3) Apply parameter update:
   `p -= lr * ( m_hat / (sqrt(v_hat) + eps) + weight_decay * p )`

Notes:
- This update is **in-place** (mutates `p`).
- Gradients should not be modified.
- State tensors must match parameter shape/device/dtype.

In [ ]:
@dataclass
class AdamWState:
    """
    Per-parameter AdamW state.

    m: first moment
    v: second moment
    t: step count
    """

    m: torch.Tensor
    v: torch.Tensor
    t: int


def init_adamw_state(p: torch.Tensor) -> AdamWState:
    """
    Initialize AdamW state tensors for a parameter tensor p.

    What to create:
    - m: zeros like p, same shape/device/dtype
    - v: zeros like p, same shape/device/dtype
    - t: step counter starting at 0

    Notes / requirements:
    - Use torch.zeros_like(p) for m and v.
    - Do NOT attach gradients to the state (initialize under torch.no_grad()).
    - t starts at 0. In adamw_step_, increment t to 1 on the first update *before*
      computing bias correction terms (1 - beta1^t) and (1 - beta2^t).
    - State tensors must live on the same device as p (CPU vs GPU) and have the
      same dtype as p.
    """
    with torch.no_grad():
      m = torch.zeros_like(p)
      v = torch.zeros_like(p)
   
    return AdamWState(m, v, t=0)

    

In [41]:
def adamw_step_(
    p: torch.Tensor,
    grad: torch.Tensor,
    state: AdamWState,
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> AdamWState:
    """
    In-place AdamW parameter update (updates p).

    Algorithm (AdamW):
      m = beta1*m + (1-beta1)*grad
      v = beta2*v + (1-beta2)*grad^2
      m_hat = m / (1 - beta1^t)
      v_hat = v / (1 - beta2^t)
      p = p - lr * (m_hat / (sqrt(v_hat) + eps) + weight_decay * p)

    Requirements:
    - Update p in-place.
    - Return updated state (with incremented t).
    - Do not modify grad.
    - Should work for any tensor shape.
    """
    beta1, beta2 = betas
    t = state.t + 1
    m = beta1 * state.m  + (1-beta1)*grad
    v = beta2 * state.v + (1-beta2) * grad**2
    m_hat = m / (1 - beta1**t)
    v_hat = v / (1 - beta2**t)
    p -=  lr * (m_hat / (torch.sqrt(v_hat) + eps) + weight_decay * p)
    return AdamWState(m, v, t)

def test_adamw_step_one_update():
    p = torch.tensor([1.0, -2.0])
    grad = torch.tensor([0.1, -0.2])
    state = init_adamw_state(p)

    old_p = p.clone()
    old_grad = grad.clone()

    lr = 0.01
    betas = (0.9, 0.999)
    eps = 1e-8
    weight_decay = 0.01

    new_state = adamw_step_(
        p,
        grad,
        state,
        lr=lr,
        betas=betas,
        eps=eps,
        weight_decay=weight_decay,
    )

    beta1, beta2 = betas
    expected_m = (1 - beta1) * grad
    expected_v = (1 - beta2) * grad**2
    expected_t = 1

    expected_m_hat = expected_m / (1 - beta1**expected_t)
    expected_v_hat = expected_v / (1 - beta2**expected_t)

    expected_p = old_p - lr * (
        expected_m_hat / (torch.sqrt(expected_v_hat) + eps)
        + weight_decay * old_p
    )

    assert new_state.t == expected_t
    assert torch.allclose(new_state.m, expected_m)
    assert torch.allclose(new_state.v, expected_v)
    assert torch.allclose(p, expected_p)

    # grad should not be modified
    assert torch.equal(grad, old_grad)

    # parameter should have changed in-place
    assert not torch.equal(p, old_p)

print(test_adamw_step_one_update())
    


None


In [47]:
def adamw_step_many_(
    params: list[torch.Tensor],
    grads: list[torch.Tensor],
    states: list[AdamWState],
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> list[AdamWState]:
    """
    Apply AdamW to many parameters.

    Requirements:
    - len(params) == len(grads) == len(states)
    - Update each param in-place.
    - Return the list of updated states.
    """
    updates = []
    for param, grad, state in zip(params, grads, states):
        updates.append(adamw_step_(param, grad, state, lr, betas, eps, weight_decay))
    return updates

def test_adamw():
    params = [
        torch.tensor([1.0, -2.0]),
        torch.tensor([[0.5, -1.5], [2.0, -3.0]]),
    ]
    grads = [
        torch.tensor([0.1, -0.2]),
        torch.tensor([[0.05, -0.1], [0.2, -0.3]]),
    ]
    states = [init_adamw_state(p) for p in params]

    # copies for reference computation
    ref_params = [p.clone() for p in params]
    ref_grads = [g.clone() for g in grads]
    ref_states = [init_adamw_state(p) for p in ref_params]

    lr = 0.01
    betas = (0.9, 0.999)
    eps = 1e-8
    weight_decay = 0.01

    # run batched version
    new_states = adamw_step_many_(
        params,
        grads,
        states,
        lr=lr,
        betas=betas,
        eps=eps,
        weight_decay=weight_decay,
    )

    # run individual version as ground truth
    expected_states = []
    for p, g, s in zip(ref_params, ref_grads, ref_states):
        s_new = adamw_step_(
            p,
            g,
            s,
            lr=lr,
            betas=betas,
            eps=eps,
            weight_decay=weight_decay,
        )
        expected_states.append(s_new)

    # compare parameters
    for p, p_expected in zip(params, ref_params):
        assert torch.allclose(p, p_expected)

    # compare states
    for s, s_expected in zip(new_states, expected_states):
        assert s.t == s_expected.t
        assert torch.allclose(s.m, s_expected.m)
        assert torch.allclose(s.v, s_expected.v)
print(test_adamw())


None


## Training basics

A minimal training step follows the same pattern almost everywhere:

1) set model to train mode
2) reset gradients
3) forward pass
4) compute loss
5) backward pass
6) step optimizer

In this exercise you’ll implement a single MSE training step using a standard PyTorch optimizer.
Return a Python float loss value.

In [51]:
def train_step_mse(
    model: nn.Module,
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
) -> float:
    """
    One MSE train step using standard torch optimizer.
    """
    running_loss = 0.0
    criterion = nn.MSELoss()
    model.train()
  
    x,y = batch      
    optimizer.zero_grad()
    pred = model(x)
    loss = criterion(pred, y)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()
    
    return running_loss

def test_train_step():
    torch.manual_seed(0)

    model = nn.Linear(3, 1)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

    x = torch.tensor([[1.0, 2.0, 3.0],
                      [0.5, 1.0, -1.0]])
    y = torch.tensor([[1.0],
                      [0.0]])

    old_params = [p.detach().clone() for p in model.parameters()]

    loss = train_step_mse(model, (x, y), optimizer)

    assert isinstance(loss, float)

    new_params = list(model.parameters())
    assert any(not torch.allclose(old, new) for old, new in zip(old_params, new_params))
print(test_train_step())

None


## Parameter initialization

Initialization matters because it controls signal and gradient scales at the start of training.

### Fan-in / fan-out
- `fan_in`: number of input connections to a unit
- `fan_out`: number of output connections from a unit

For a Linear layer weight of shape `(out_features, in_features)`:
- `fan_in = in_features`
- `fan_out = out_features`

### Common schemes
- **Xavier / Glorot** (often good for tanh / linear-ish nets):
  keeps variance stable across layers when activations are roughly symmetric.
- **Kaiming / He** (often good for ReLU-like nets):
  accounts for the fact that ReLU zeroes out about half the inputs.

In this section you’ll implement Xavier uniform and Kaiming uniform and use them to initialize `nn.Linear`.
We also always zero the bias unless explicitly told otherwise.

In [56]:
def fan_in_fan_out(weight: torch.Tensor) -> tuple[int, int]:
    """Compute (fan_in, fan_out) for a weight tensor."""
    size = weight.size()
    return (size[1], size[0])

def test_fan_in_fan_out_linear_weight():
    weight1 = torch.empty(3, 5)
    fan_in1, fan_out1 = fan_in_fan_out(weight1)
    assert fan_in1 == 5
    assert fan_out1 == 3

    weight2 = torch.empty(10, 4)
    fan_in2, fan_out2 = fan_in_fan_out(weight2)
    assert fan_in2 == 4
    assert fan_out2 == 10

print(test_fan_in_fan_out_linear_weight())

None


In [60]:
import math
def xavier_uniform_(weight: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    """
    In-place Xavier/Glorot uniform init:
      bound = gain * sqrt(6 / (fan_in + fan_out))
      U(-bound, bound)
    """
    torch.manual_seed(0)
    fan_in, fan_out = fan_in_fan_out(weight)
    bound = gain * math.sqrt(6 / (fan_in + fan_out))
    
    with torch.no_grad():
      return weight.uniform_(-bound, bound)
def test_xavier_uniform_bounds():
    torch.manual_seed(0)

    weight = torch.empty(3, 5)
    out = xavier_uniform_(weight, gain=1.0)

    bound = math.sqrt(6.0 / (5 + 3))

    assert out is weight
    assert torch.all(weight >= -bound)
    assert torch.all(weight <= bound)
    assert not torch.all(weight == 0)
print(test_xavier_uniform_bounds())

None


In [63]:
def kaiming_uniform_(weight: torch.Tensor, nonlinearity: str = "relu") -> torch.Tensor:
    """
    In-place Kaiming/He uniform init.

    Follow this common choice:
      gain = sqrt(2) for ReLU
      std = gain / sqrt(fan_in)
      bound = sqrt(3) * std
      U(-bound, bound)
    """
    gain = 1
    fan_in, fan_out = fan_in_fan_out(weight)
    if nonlinearity == "relu":
        gain = math.sqrt(2)
    
    std = gain / math.sqrt(fan_in)
    bound = math.sqrt(3) * std
        
    return weight.uniform_(-bound, bound)

def test_kaiming():
    torch.manual_seed(0)

    weight = torch.empty(3, 5)
    original_ptr = weight.data_ptr()

    out = kaiming_uniform_(weight)

    fan_in = 5
    expected_bound = math.sqrt(6.0 / fan_in)

    assert out is weight
    assert weight.data_ptr() == original_ptr
    assert weight.shape == (3, 5)
    assert torch.all(weight >= -expected_bound)
    assert torch.all(weight <= expected_bound)
    assert not torch.all(weight == 0)

    
print(test_kaiming())

None


In [ ]:
def init_linear_(layer: nn.Linear, scheme: str = "xavier") -> nn.Linear:
    """
    Initialize an nn.Linear in-place.

    scheme:
      - "xavier"
      - "kaiming_relu"
      - "zero" (weights and bias =  0)
    """
    
    with torch.no_grad():
        if scheme == "xavier":
            xavier_uniform_(layer.weight, gain=1.0)
        elif scheme == "kaiming_relu":
            kaiming_uniform_(layer.weight, nonlinearity="relu")
        elif scheme == "zero":
            layer.weight.zero_()
            layer.bias.zero_()
        else:
            raise ValueError(f"Unknown scheme: {scheme}")
        if layer.bias is not None:
            layer.bias.zero_()

    return layer


def test_init_linear():
    torch.manual_seed(0)

    layer = nn.Linear(5, 3, bias=True)
    weight_ptr = layer.weight.data_ptr()
    bias_ptr = layer.bias.data_ptr()

    out = init_linear_(layer, scheme="xavier")

    expected_bound = math.sqrt(6.0 / (5 + 3))

    assert out is layer
    assert layer.weight.data_ptr() == weight_ptr
    assert layer.bias.data_ptr() == bias_ptr

    assert torch.all(layer.weight >= -expected_bound)
    assert torch.all(layer.weight <= expected_bound)
    assert torch.all(layer.bias == 0)
print(test_init_linear())

None
